## Import Files

In [1]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
import pandas as pd
import numpy as np
import json
from mstrio.connection import Connection
from mstr_robotics.osi_expoter import export_dashboard
from mstr_robotics.dossier import doss_read_out, doss_read_out_det
with open('..\\config\\user_d.json', 'r', encoding='utf-8') as openfile:
    user_d = json.load(openfile)

with open(OSI_SCHEMA, 'r', encoding='utf-8') as openfile:
    osi_dashboard_schema_d = json.load(openfile)

## Notebook code

## Ontology Mapper

In [ ]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
import pandas as pd
from pathlib import Path


def add_ontology_col(table, primary_key, excel_file, output_path):
    df    = pd.read_excel(excel_file)
    lines = []

    # ── 1. ALTER TABLE ────────────────────────────────────────────────────────
    lines.append(f"-- Add ontology columns to {table}")
    lines.append(f"ALTER TABLE {table} ADD COLUMN ontology_id VARCHAR(500);\n")
    lines.append(f"ALTER TABLE {table} ADD COLUMN ontology_uri VARCHAR(500);\n")

    # ── 2. UPDATE statements ──────────────────────────────────────────────────
    lines.append(f"-- Populate ontology from Wikidata columns")
    for _, row in df.iterrows():
        if pd.isna(row[primary_key]):
            continue
        row_id = int(row[primary_key])

        wiki_id  = str(row["Wikidata_ID"]).strip()
        wiki_uri = str(row["Wikidata_URI"]).strip()
        if wiki_id in ("no direct Wikidata match", "--", "", "nan"):
            wiki_id  = "No Ontology available"
            wiki_uri = "No_ontology"

        ontology_id_escaped = wiki_id.replace("'", "''")
        ontology_uri_escaped = wiki_uri.replace("'", "''")

        lines.append(
            f"UPDATE {table} SET "
            f"ontology_id = '{ontology_id_escaped}' "
            f", ontology_uri = '{ontology_uri_escaped}' "
            f"WHERE {primary_key} = {row_id};"
        )

    sql = "\n".join(lines)
    Path(output_path).write_text(sql, encoding="utf-8")
    print(f"Written {len(df)} UPDATE statements → {output_path}")
    print("\nPreview (first 7 lines):")
    for line in sql.splitlines()[:7]:
        print(" ", line)


# ── run for brands ────────────────────────────────────────────────────────────
add_ontology_col(
    table       = "LU_CUSTOMER",
    primary_key = "zipcode",
    excel_file  = r"C:\Users\danie\Downloads\URI_Zip_Code.xlsx",
    output_path = str(REPO_ROOT / "import_files" / "update_zipcode_ontology.sql"),
)
